<a href="https://colab.research.google.com/github/soberbichler/Bring-your-own-data-Lab_LLMs/blob/main/Example_Fine_Tuning_Llama_HF_Job.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Adapter-Based Tuning
##Llama 3.1 8B Base Model

Adapter-based tuning, inlike fine-tuning, modifies only a small subset of model parameters by inserting lightweight modules between layers while keeping original weights frozen. Methods like LoRA add low-rank matrices to approximate fine-tuning, requiring updates to just 1 % of parameters in large models. This approach reduces computational costs and enables multi-task efficiency—one base model can host multiple task-specific adapters, eliminating the need for separate full models for different tasks like translation or summarization.

In [ ]:
#Install and import required packages
!pip install pandas openpyxl scikit-learn ftfy
!pip install -q --upgrade transformers huggingface_hub
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q -U transformers huggingface_hub bitsandbytes accelerate peft

import pandas as pd
import json
import re
from sklearn.model_selection import train_test_split
import os
import ftfy  # Fixes text encoding issues
import random
from collections import Counter

## First, we need to prepare our dataset to create Llama training examples

Preparing a dataset for Llama training involves several crucial steps to transform raw data into the specific format that Llama models expect. The process begins with collecting and cleaning your source data, removing any unwanted artifacts, duplicates, or formatting inconsistencies that could negatively impact training quality. Next, the data must be tokenized using Llama's specific tokenizer to ensure compatibility with the model's vocabulary and token limits. Each training example typically needs to be structured with appropriate prompt templates, often including system prompts, user inputs, and assistant responses clearly delineated with special tokens like <|begin_of_text|>, <|start_header_id|>, and <|eot_id|>. For fine-tuning tasks, the dataset should be formatted as instruction-response pairs or conversation chains, with careful attention paid to maintaining consistent formatting throughout. Additionally, it's important to split the data into training and validation sets, ensure proper text encoding (UTF-8), and consider implementing data augmentation strategies if your dataset is limited in size. The final prepared dataset should be saved in a format compatible with your training framework, such as JSON, JSONL, or Hugging Face's dataset format, ready to be loaded efficiently during the training process.

In [ ]:
import json, random

INSTRUCTION = "Extract the knowledge graph triple from the argument. Return JSON with keys: subject, predicate, object, category."

# 3 examples
examples = [
    {
        "instruction": INSTRUCTION,
        "input": (
            "<argument>Die Geologen sagen, das Erdbeben sei tektonischen Charakters "
            "wie jenes von 1905.</argument>"
            "<claim>The earthquakes follow a recurring tectonic pattern.</claim>"
            "<explanation>Scientists compare current seismic activity to historical data.</explanation>"
        ),
        "output": '{"subject": "scientific", "predicate": "explains", "object": "damage", "category": "scientific"}',
    },
    {
        "instruction": INSTRUCTION,
        "input": (
            "<argument>Das Militär hat die Hilfslieferungen verzögert.</argument>"
            "<claim>Military interference slowed down aid distribution.</claim>"
            "<explanation>The argument criticizes military obstruction of relief efforts.</explanation>"
        ),
        "output": '{"subject": "military", "predicate": "delays", "object": "aid", "category": "military"}',
    },
    {
        "instruction": INSTRUCTION,
        "input": (
            "<argument>Die Gesellschaft ist verpflichtet, den Opfern zu helfen.</argument>"
            "<claim>Society has a moral obligation to support disaster victims.</claim>"
            "<explanation>The argument frames aid as a societal duty rather than a choice.</explanation>"
        ),
        "output": '{"subject": "society", "predicate": "is obligated to", "object": "aid", "category": "aid"}',
    },
]

# Train / validation split (2 train, 1 validation)
random.seed(42)
random.shuffle(examples)

split = int(len(examples) * 0.67)
train = examples[:split]
val   = examples[split:]

# Save
for split_name, data in [("train", train), ("validation", val)]:
    path = f"{split_name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for r in data:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"💾 {path}: {len(data)} record(s)")

# Preview
print("\n--- train.jsonl ---")
for r in train:
    print(json.dumps(r, indent=2, ensure_ascii=False))

print("\n--- validation.jsonl ---")
for r in val:
    print(json.dumps(r, indent=2, ensure_ascii=False))

#Ready for Training

In [ ]:
import os, json

files_to_check = ["train.jsonl", "validation.jsonl"]

for file in files_to_check:
    if os.path.exists(file):
        with open(file, "r", encoding="utf-8") as f:
            lines = f.readlines()
        first = json.loads(lines[0])
        print(f"✓ {file}: {len(lines)} examples, {os.path.getsize(file):,} bytes")
        print(f"  instruction : {first['instruction'][:60]}...")
        print(f"  input       : {first['input'][:60]}...")
        print(f"  output      : {first['output']}")
        print()
    else:
        print(f"✗ {file} not found!")

In [ ]:
# 2: Install HF hub
!pip install -U huggingface_hub

In [ ]:
# 3: Set your HF username and dataset repo
import os
os.environ['HF_USERNAME'] = 'oberbics'
os.environ['HF_DATASET_REPO'] = 'oberbics/jobs'
print(f"Username: {os.environ['HF_USERNAME']}")
print(f"Dataset repo: {os.environ['HF_DATASET_REPO']}")

In [ ]:
# 4: Login to Hugging Face
# You'll need your HF token from https://huggingface.co/settings/tokens
!pip install huggingface_hub==0.26.5 -q
!huggingface-cli login

## Create Training Script

This script is designed for adapter-based tuning of Meta's Llama 3.1 8B base model for argumentative unit extraction. The script loads training data from JSON files containing text examples, then adapts the large language model to extract units in a structured format (argument, claim, explanation and verification for expert-in-the-loop control).

The technical implementation uses several memory optimization techniques to handle the 8B parameter model efficiently. It employs 4-bit quantization through BitsAndBytesConfig to reduce memory usage, implements LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning, and uses gradient checkpointing along with CPU/disk offloading to manage GPU memory constraints. The script is configured to work with limited hardware by using techniques like mixed precision training (fp16) and paged AdamW optimizer, making it feasible to train on systems with memory limitations while still working with the full Llama 3.1 model.

The training configuration includes several anti-overfitting measures (it is more important to force the model into the xml structure than learning from the examples) that were specifically updated in this version of the script. It reduces the LoRA rank from 16 to 8, increases dropout from 0.1 to 0.2, decreases the learning rate to 5e-5, and limits training to 2 epochs instead of 3. Additional regularization techniques include weight decay (L2 regularization) and label smoothing to prevent the model from becoming overconfident. The script also includes validation monitoring if a validation dataset is available, and implements comprehensive logging and checkpointing to track training progress and save the model at regular intervals.

In [ ]:
llama_31_base_kg_script = """
import os
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login
import gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
token = os.environ.get("HF_TOKEN")
login(token=token)

print("Loading dataset...")
train_dataset = load_dataset("json", data_files="train.jsonl", split="train")
try:
    val_dataset = load_dataset("json", data_files="validation.jsonl", split="train")
    has_validation = True
except:
    val_dataset = None
    has_validation = False

print(f"✓ Training examples: {len(train_dataset)}")
if has_validation:
    print(f"✓ Validation examples: {len(val_dataset)}")

torch.cuda.empty_cache()
gc.collect()

model_id = "meta-llama/Llama-3.1-8B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    offload_folder="./offload",
    offload_state_dict=True,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

def tokenize_function(examples):
    texts = [
        f"### Instruction:\\n{inst}\\n\\n### Input:\\n{inp}\\n\\n### Response:\\n{out}"
        for inst, inp, out in zip(
            examples["instruction"],
            examples["input"],
            examples["output"]
        )
    ]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=2048)

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
if has_validation:
    val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

training_args = TrainingArguments(
    output_dir="./llama-3.1-base-kg-extraction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    warmup_steps=50,
    learning_rate=3e-5,
    fp16=True,
    logging_steps=5,
    save_steps=50,
    save_total_limit=3,
    eval_strategy="steps" if has_validation else "no",
    eval_steps=50 if has_validation else None,
    load_best_model_at_end=True if has_validation else False,
    metric_for_best_model="loss" if has_validation else None,
    greater_is_better=False if has_validation else None,
    push_to_hub=True,
    hub_model_id="oberbics/llama-3.1-base-kg-extraction",
    hub_token=token,
    report_to=[],
    optim="paged_adamw_8bit",
    max_grad_norm=0.5,
    warmup_ratio=0.05,
    weight_decay=0.0,
    label_smoothing_factor=0.0,
    lr_scheduler_type="cosine"
)

callbacks = []
if has_validation:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized if has_validation else None,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=callbacks
)

print("\\nStarting KG extraction training on base model...")
trainer.train()
trainer.save_model()
tokenizer.push_to_hub("oberbics/llama-3.1-base-kg-extraction", token=token)
print("✅ Base model saved to Hugging Face Hub")
"""

with open("llama_31_base_kg.py", "w") as f:
    f.write(llama_31_base_kg_script)
print("Created llama_31_base_kg.py")

In [ ]:

!pip install huggingface_hub --upgrade -q

from huggingface_hub import HfApi, get_token
api = HfApi()

# Upload training script
api.upload_file(
    path_or_fileobj="llama_31_base_kg.py",
    path_in_repo="kg-extraction/llama_31_base_kg.py",
    repo_id="oberbics/jobs",
    repo_type="dataset"
)

# Upload data files
api.upload_file(
    path_or_fileobj="train.jsonl",
    path_in_repo="kg-extraction/train.jsonl",
    repo_id="oberbics/jobs",
    repo_type="dataset"
)

api.upload_file(
    path_or_fileobj="validation.jsonl",
    path_in_repo="kg-extraction/validation.jsonl",
    repo_id="oberbics/jobs",
    repo_type="dataset"
)

# Get token
token = get_token()

# Launch command
command = f"""echo 'Starting Base Model KG Training' && \
export HF_TOKEN='{token}' && \
apt-get update -qq && apt-get install -y -qq wget && \
pip install -q torch transformers peft datasets bitsandbytes accelerate huggingface_hub && \
wget -q https://huggingface.co/datasets/oberbics/jobs/resolve/main/kg-extraction/train.jsonl && \
wget -q https://huggingface.co/datasets/oberbics/jobs/resolve/main/kg-extraction/validation.jsonl && \
wget -q https://huggingface.co/datasets/oberbics/jobs/resolve/main/kg-extraction/llama_31_base_kg.py && \
python llama_31_base_kg.py"""

print("Launching base model KG training...")
!python -m huggingface_hub.cli.hf jobs run --flavor a100-large pytorch/pytorch:2.1.0-cuda12.1-cudnn8-devel -- /bin/bash -c "{command}"


## Merge LoRA adapter with Llama base model


In [ ]:
# 1: Install and setup
!pip install -q huggingface_hub transformers peft accelerate

from huggingface_hub import login
from google.colab import userdata
import os

# Login to Hugging Face
token = userdata.get("HF_TOKEN")  # or paste manually instead of userdata
os.environ["HF_TOKEN"] = token
login(token=token)
print("Logged in to Hugging Face")


This script performs model merging to combine the fine-tuned LoRA adapter with the original Llama 3.1 base model, creating a standalone full model. After the previous training script created a LoRA adapter (which only stores the small additional weights learned during fine-tuning), this merge script loads both the original Meta-Llama-3.1-8B-Instruct model and the trained adapter, then combines them into a single complete model that contains all the newspaper argument analysis capabilities without requiring the PEFT library.

The process involves loading the base model, applying the LoRA adapter weights to it, and then "merging and unloading" to create a standard transformer model with all weights integrated. The merged model is then uploaded to Hugging Face Hub, making it easier to use since it no longer needs separate adapter loading. This merged version can be used directly like any standard language model, while retaining all the specialized training for newspaper discourse analysis that was learned during the fine-tuning process.

In [ ]:
merge_script = """
import os, warnings
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login, HfApi

warnings.filterwarnings("ignore")

token = os.environ.get("HF_TOKEN")
print(f"Token prefix: {token[:10]}...")
login(token=token)

print("="*50)
print("Starting Model Merge: Llama 3.1 Base KG Extraction")
print("="*50)

base_model_id = "meta-llama/Llama-3.1-8B"
adapter_id    = "oberbics/llama-3.1-base-kg-extraction"
repo_id       = "oberbics/llama-3.1-base-kg-extraction-full"

# Create repo if it doesn't exist
api = HfApi()
try:
    api.create_repo(repo_id=repo_id, token=token, exist_ok=True)
    print(f"✓ Repo ready: {repo_id}")
except Exception as e:
    print(f"Repo note: {e}")

# Load base model
print(f"\\n Loading base model: {base_model_id}")
base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=token,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
print("✓ Base model loaded")

# Load adapter
print(f"\\n Loading adapter: {adapter_id}")
model_with_adapter = PeftModel.from_pretrained(
    base,
    adapter_id,
    token=token,
    device_map="auto"
)
print("✓ Adapter loaded")

# Merge
print("\\n Merging models...")
merged_model = model_with_adapter.merge_and_unload()
print("✓ Merge complete")

# Tokenizer
print("\\n Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(adapter_id, token=token)
print("✓ Tokenizer loaded")

# Save locally then upload
print("\\n Saving merged model locally...")
merged_model.save_pretrained("./merged", max_shard_size="5GB")
tokenizer.save_pretrained("./merged")
print("✓ Saved locally")

# Upload
print("\\n Uploading to Hub...")
api.upload_folder(
    folder_path="./merged",
    repo_id=repo_id,
    token=token,
    repo_type="model"
)
print("✓ Uploaded")

print("\\n" + "="*50)
print("SUCCESS!")
print(f"Full model: https://huggingface.co/{repo_id}")
print("="*50)
"""

with open("merge_llama31_base_kg.py", "w") as f:
    f.write(merge_script)
print("Created merge_llama31_base_kg.py")

In [ ]:
api = HfApi()
api.upload_file(
    path_or_fileobj="merge_llama31_base_kg.py",
    path_in_repo="merge_jobs/merge_llama31_base_kg.py",
    repo_id="oberbics/jobs",
    repo_type="dataset",
    token=token
)
print("Merge script uploaded")

In [ ]:
import subprocess

bash_command = f"""apt-get update && apt-get install -y git && \
pip install -q --upgrade pip && \
pip install -q torch transformers peft accelerate huggingface_hub bitsandbytes && \
git clone https://huggingface.co/datasets/oberbics/jobs jobs_repo && \
cd jobs_repo && \
export HF_TOKEN={token} && \
python merge_jobs/merge_llama31_base_kg.py"""

command = f"""python -m huggingface_hub.cli.hf jobs run --flavor a100-large pytorch/pytorch:2.6.0-cuda12.4-cudnn9-devel -- /bin/bash -c "{bash_command}" """

print("Launching merge job...")
result = subprocess.run(command, shell=True, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)